
Ce notebook réalise uniquement l'acquisition et la compréhension du dataset
`michaelozon/candidate-matching-synthetic`.

## 1. Imports et paramètres reproductibles

Les chemins de sortie restent relatifs au dépôt. Le code fonctionne depuis la racine du projet
ou depuis le dossier `notebooks`.

In [1]:
from collections import Counter
from pathlib import Path
from pprint import pprint
import json

import pandas as pd
from datasets import load_dataset

DATASET_ID = "michaelozon/candidate-matching-synthetic"
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

## 2. Chargement des trois tables

In [2]:
def load_table(data_dir):
    return load_dataset(
        DATASET_ID,
        data_dir=data_dir,
        split="train",
    ).to_pandas()


resumes_df = load_table("resumes")
jobs_df = load_table("jobs")
matches_df = load_table("matches")

tables = {
    "resumes": resumes_df,
    "jobs": jobs_df,
    "matches": matches_df,
}

dimensions = pd.DataFrame(
    [{"table": name, "rows": len(df), "columns": df.shape[1]} for name, df in tables.items()]
).set_index("table")
display(dimensions)

Generating train split: 0 examples [00:00, ? examples/s]

jobs/train-00000-of-00001.parquet:   0%|          | 0.00/56.0k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

matches/train-00000-of-00001.parquet:   0%|          | 0.00/274k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

,rows,columns
table,,
resumes,10000,9
jobs,2500,9
matches,2500,2


## 3. Inspection générale

Pour chaque table, on examine les colonnes, les types pandas, les cinq premières lignes et un
enregistrement complet lisible. Cette inspection sert à comprendre le schéma avant tout calcul.

In [3]:
for table_name, df in tables.items():
    print(f"\n{'=' * 80}\nTABLE : {table_name.upper()} — shape={df.shape}\n{'=' * 80}")
    print("Colonnes :", df.columns.tolist())
    print("\nTypes :")
    display(df.dtypes.rename("dtype").to_frame())
    print("5 premières lignes :")
    display(df.head())
    print("Exemple complet (première ligne) :")
    pprint(df.iloc[0].to_dict(), sort_dicts=False)


TABLE : RESUMES — shape=(10000, 9)
Colonnes : ['resume_id', 'role', 'seniority', 'years_experience', 'industry', 'education', 'skills', 'summary', 'experience_bullets']

Types :


,dtype
resume_id,object
role,object
seniority,object
years_experience,int64
industry,object
education,object
skills,object
summary,object
experience_bullets,object


5 premières lignes :


,resume_id,role,seniority,years_experience,industry,education,skills,summary,experience_bullets
0,R_000000,Software Engineer,Senior,12,EdTech,BSc,"[OOP, Databases, Git, Docker, Python, Unit Testing, Java]",Software Engineer with 12 years of experience in EdTech.,"[Delivered results using structured workflows and clear communication, Collaborated with stakeholders to define need..."
1,R_000001,Marketing Manager,Junior,2,E-commerce,BSc,"[Content Marketing, Meta Ads, Conversion Optimization, SEO, Landing Pages]",Marketing Manager with 2 years of experience in E-commerce.,"[Delivered results using structured workflows and clear communication, Collaborated with stakeholders to define need..."
2,R_000002,Full Stack Engineer,Mid,6,FinTech,BA,"[Databases, OOP, Git, Docker, JavaScript, Python]",Full Stack Engineer with 6 years of experience in FinTech.,"[Delivered results using structured workflows and clear communication, Collaborated with stakeholders to define need..."
3,R_000003,Data Analyst,Junior,0,Healthcare,BSc,"[Excel, A/B Testing, Tableau, Pandas, Power BI, Data Visualization]",Data Analyst with 0 years of experience in Healthcare.,"[Delivered results using structured workflows and clear communication, Collaborated with stakeholders to define need..."
4,R_000004,BI Analyst,Junior,0,Gaming,BA,"[Statistics, SQL, Power BI, Pandas, Excel]",BI Analyst with 0 years of experience in Gaming.,"[Delivered results using structured workflows and clear communication, Collaborated with stakeholders to define need..."


Exemple complet (première ligne) :
{'resume_id': 'R_000000',
 'role': 'Software Engineer',
 'seniority': 'Senior',
 'years_experience': 12,
 'industry': 'EdTech',
 'education': 'BSc',
 'skills': array(['OOP', 'Databases', 'Git', 'Docker', 'Python', 'Unit Testing',
       'Java'], dtype=object),
 'summary': 'Software Engineer with 12 years of experience in EdTech.',
 'experience_bullets': array(['Delivered results using structured workflows and clear communication',
       'Collaborated with stakeholders to define needs and execute tasks',
       'Maintained reporting and documentation to support team performance'],
      dtype=object)}

TABLE : JOBS — shape=(2500, 9)
Colonnes : ['job_id', 'job_title', 'seniority', 'industry', 'must_have_skills', 'nice_to_have_skills', 'description', 'responsibilities', 'requirements']

Types :


...

Exemple complet (première ligne) :
{'job_id': 'J_000000',
 'job_title': 'Project Manager',
 'seniority': 'Senior',
 'industry': 'Retail',
 'must_have_skills': array(['Timeline Management', 'Process Improvement',
       'Stakeholder Communication'], dtype=object),
 'nice_to_have_skills': array(['Asana'], dtype=object),
 'description': 'We are hiring a Project Manager to support teams in Retail.',
 'responsibilities': array(['Deliver high-quality work aligned with goals and timelines',
       'Collaborate with stakeholders and communicate progress clearly',
       'Improve processes and contribute to measurable outcomes'],
      dtype=object),
 'requirements': array(['Relevant experience and ability to learn quickly',
       'Strong communication and ownership mindset',
       'Comfort working with tools, data, and cross-functional teams'],
      dtype=object)}

TABLE : MATCHES — shape=(2500, 2)
Colonnes : ['job_id', 'relevant_resume_ids']

Types :


,dtype
job_id,object
relevant_resume_ids,object


5 premières lignes :


,job_id,relevant_resume_ids
0,J_000000,"[R_000915, R_003468, R_003641, R_003326, R_008830, R_008352, R_005021, R_004675, R_009452, R_007618, R_002906, R_004..."
1,J_000001,"[R_003953, R_009866, R_009933, R_009826, R_007936, R_000313, R_003819, R_009170, R_009301, R_005820, R_006132, R_001..."
2,J_000002,"[R_001833, R_005742, R_001158, R_000978, R_007579, R_000949, R_009313, R_000221, R_004203, R_008213, R_006191, R_005..."
3,J_000003,"[R_002054, R_008377, R_003385, R_001576, R_006095, R_000835, R_008183, R_001043, R_004094, R_009258, R_008149, R_009..."
4,J_000004,"[R_002908, R_003365, R_008335, R_005987, R_006550, R_008237, R_006290, R_002333, R_003323, R_004592, R_001502, R_001..."


Exemple complet (première ligne) :
{'job_id': 'J_000000',
 'relevant_resume_ids': array(['R_000915', 'R_003468', 'R_003641', 'R_003326', 'R_008830',
       'R_008352', 'R_005021', 'R_004675', 'R_009452', 'R_007618',
       'R_002906', 'R_004328', 'R_007632', 'R_004200', 'R_003626',
       'R_007572', 'R_002900', 'R_003663', 'R_004748', 'R_004905',
       'R_003787', 'R_009660', 'R_006924', 'R_005025', 'R_009156',
       'R_003647', 'R_005716', 'R_001302', 'R_008297', 'R_000201'],
      dtype=object)}


## 4. Comptages fondamentaux

Une ligne de `matches` décrit un Job et sa liste de CV pertinents. Le nombre de couples 
est donc la somme des longueurs de `relevant_resume_ids`.

In [4]:
def as_list(value):
    if value is None:
        return []
    if isinstance(value, (list, tuple)):
        return list(value)
    if hasattr(value, "tolist"):
        return value.tolist()
    return []


match_resume_lists = matches_df["relevant_resume_ids"].map(as_list)
n_resumes = len(resumes_df)
n_jobs = len(jobs_df)
n_match_records = len(matches_df)
n_positive_pairs = int(match_resume_lists.map(len).sum())

fundamental_counts = pd.Series({
    "Nombre de CV": n_resumes,
    "Nombre de Jobs": n_jobs,
    "Nombre de lignes dans matches": n_match_records,
    "Nombre total de couples CV–Job positifs": n_positive_pairs,
}, name="valeur")
display(fundamental_counts.to_frame())

,valeur
Nombre de CV,10000
Nombre de Jobs,2500
Nombre de lignes dans matches,2500
Nombre total de couples CV–Job positifs,75000


## 5. Vérification des identifiants

On contrôle l'unicité des clés principales et l'absence d'identifiants nuls ou constitués
uniquement d'espaces. Aucun correctif n'est appliqué.

In [5]:
def blank_id_count(series):
    return int(series.isna().sum() + series.dropna().astype(str).str.strip().eq("").sum())


id_checks = pd.Series({
    "resume_id uniques": bool(resumes_df["resume_id"].is_unique),
    "job_id uniques": bool(jobs_df["job_id"].is_unique),
    "resume_id vides": blank_id_count(resumes_df["resume_id"]),
    "job_id vides": blank_id_count(jobs_df["job_id"]),
}, name="résultat")
display(id_checks.to_frame())

,résultat
resume_id uniques,True
job_id uniques,True
resume_id vides,0
job_id vides,0


## 6. Intégrité référentielle

Les identifiants utilisés par `matches` sont comparés aux clés des tables de référence. Les
compteurs ci-dessous portent sur les identifiants distincts invalides, et le détail est conservé
pour faciliter le diagnostic.

In [6]:
resume_ids = set(resumes_df["resume_id"].dropna())
job_ids = set(jobs_df["job_id"].dropna())
match_job_ids = set(matches_df["job_id"].dropna())
referenced_resume_ids = {rid for ids in match_resume_lists for rid in ids if rid is not None}

invalid_job_ids = sorted(match_job_ids - job_ids)
invalid_resume_ids = sorted(referenced_resume_ids - resume_ids)
invalid_job_references = len(invalid_job_ids)
invalid_resume_references = len(invalid_resume_ids)

integrity = pd.Series({
    "Références job_id invalides": invalid_job_references,
    "Références resume_id invalides": invalid_resume_references,
}, name="nombre")
display(integrity.to_frame())
print("job_id invalides :", invalid_job_ids[:20])
print("resume_id invalides :", invalid_resume_ids[:20])

,nombre
Références job_id invalides,0
Références resume_id invalides,0


job_id invalides : []
resume_id invalides : []


## 7. Doublons et valeurs manquantes

Les colonnes contenant des listes sont rendues comparables uniquement pour compter les lignes
dupliquées. Les DataFrames d'origine restent inchangés. Les valeurs manquantes sont comptées par
colonne sans imputation.

In [7]:
def make_hashable(value):
    if isinstance(value, (list, tuple)):
        return tuple(make_hashable(item) for item in value)
    if hasattr(value, "tolist"):
        return tuple(make_hashable(item) for item in value.tolist())
    if isinstance(value, dict):
        return tuple(sorted((key, make_hashable(item)) for key, item in value.items()))
    return value


duplicate_rows = {}
missing_by_table = {}

for table_name, df in tables.items():
    comparable_df = df.map(make_hashable)
    duplicate_rows[table_name] = int(comparable_df.duplicated().sum())
    missing_by_table[table_name] = df.isna().sum().astype(int)

duplicate_report = pd.Series(duplicate_rows, name="lignes dupliquées")
missing_report = pd.DataFrame(missing_by_table).fillna(0).astype(int).T

display(duplicate_report.to_frame())
display(missing_report)

,lignes dupliquées
resumes,0
jobs,0
matches,0


,description,education,experience_bullets,industry,job_id,job_title,must_have_skills,nice_to_have_skills,relevant_resume_ids,requirements,responsibilities,resume_id,role,seniority,skills,summary,years_experience
resumes,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
jobs,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
matches,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 8. Compréhension des labels

Les tableaux de fréquences permettent d'identifier les catégories dominantes et leur équilibre,
sans regroupement ni recodage.

In [8]:
def show_distributions(df, columns, table_name):
    print(f"\nDistributions — {table_name}")
    for column in columns:
        print(f"\n{column}")
        counts = df[column].value_counts(dropna=False).rename("count").to_frame()
        counts["percentage"] = (100 * counts["count"] / len(df)).round(2)
        display(counts)


show_distributions(resumes_df, ["seniority", "role", "industry", "education"], "CV")
show_distributions(jobs_df, ["seniority", "industry", "job_title"], "Jobs")


Distributions — CV

seniority


,count,percentage
seniority,,
Junior,3372,33.72
Mid,3365,33.65
Senior,3263,32.63



role


,count,percentage
role,,
Sales Representative,460,4.60
Backend Engineer,456,4.56
Marketing Manager,455,4.55
Product Manager,453,4.53
Associate Product Manager,438,4.38
Financial Analyst,433,4.33
Software Engineer,432,4.32
Operations Manager,427,4.27
BI Analyst,426,4.26



industry


...

,count,percentage
seniority,,
Junior,855,34.2
Mid,825,33.0
Senior,820,32.8



industry


,count,percentage
industry,,
Healthcare,290,11.60
FinTech,279,11.16
SaaS,271,10.84
Logistics,248,9.92
Cybersecurity,247,9.88
Travel,243,9.72
Gaming,241,9.64
Retail,237,9.48
EdTech,225,9.00



job_title


,count,percentage
job_title,,
Project Manager,130,5.20
Business Analyst,120,4.80
Junior Accountant,118,4.72
Backend Engineer,117,4.68
Sales Representative,114,4.56
Technical Support Specialist,112,4.48
FP&A Analyst,111,4.44
Full Stack Engineer,108,4.32
Account Executive,108,4.32


## 9. compétences

In [9]:
resume_skill_lists = resumes_df["skills"].map(as_list)
job_skill_lists = jobs_df["must_have_skills"].map(as_list)

resume_skill_counts = Counter(skill for skills in resume_skill_lists for skill in skills)
job_skill_counts = Counter(skill for skills in job_skill_lists for skill in skills)

n_unique_resume_skills = len(resume_skill_counts)
n_unique_job_skills = len(job_skill_counts)
mean_resume_skills = resume_skill_lists.map(len).mean()
mean_job_must_have_skills = job_skill_lists.map(len).mean()

skill_summary = pd.Series({
    "Compétences uniques côté CV": n_unique_resume_skills,
    "Compétences uniques côté Jobs": n_unique_job_skills,
    "Nombre moyen de compétences par CV": mean_resume_skills,
    "Nombre moyen de must_have_skills par Job": mean_job_must_have_skills,
}, name="valeur")
display(skill_summary.to_frame())

print("20 compétences les plus fréquentes côté CV")
display(pd.DataFrame(resume_skill_counts.most_common(20), columns=["skill", "count"]))
print("20 compétences must-have les plus fréquentes côté Job")
display(pd.DataFrame(job_skill_counts.most_common(20), columns=["skill", "count"]))

,valeur
Compétences uniques côté CV,73.0000
Compétences uniques côté Jobs,73.0000
Nombre moyen de compétences par CV,6.4888
Nombre moyen de must_have_skills par Job,3.9972


20 compétences les plus fréquentes côté CV


,skill,count
0,A/B Testing,2447
1,Python,1697
2,Forecasting,1648
3,Reporting,1619
4,KPIs,1598
5,Excel,1551
6,JavaScript,866
7,OOP,858
8,Prioritization,858
9,REST APIs,855


20 compétences must-have les plus fréquentes côté Job


,skill,count
0,A/B Testing,361
1,Forecasting,277
2,Excel,265
3,Reporting,264
4,KPIs,242
5,Python,238
6,Project Planning,146
7,Valuation,146
8,Intercom,143
9,Databases,140


## 10. Analyse des matches

On décrit le nombre de CV pertinents par Job, puis on éclate temporairement les listes pour
vérifier si le même couple `(job_id, resume_id)` apparaît plusieurs fois. Aucun graphe n'est créé.

In [10]:
relevant_counts = match_resume_lists.map(len).rename("n_relevant_resumes")
match_stats = pd.Series({
    "minimum": int(relevant_counts.min()),
    "maximum": int(relevant_counts.max()),
    "moyenne": float(relevant_counts.mean()),
    "médiane": float(relevant_counts.median()),
}, name="valeur")
display(match_stats.to_frame())

print("Distribution du nombre de CV pertinents par Job")
display(
    relevant_counts.value_counts().sort_index().rename_axis("taille_liste").rename("nombre_jobs").to_frame()
)

pairs_df = pd.DataFrame(
    [(job_id, resume_id) for job_id, ids in zip(matches_df["job_id"], match_resume_lists) for resume_id in ids],
    columns=["job_id", "resume_id"],
)
n_duplicate_pairs = int(pairs_df.duplicated().sum())
print("Nombre d'occurrences de couples (job_id, resume_id) dupliquées :", n_duplicate_pairs)
if n_duplicate_pairs:
    display(pairs_df[pairs_df.duplicated(keep=False)].sort_values(["job_id", "resume_id"]).head(20))

,valeur
minimum,30.0
maximum,30.0
moyenne,30.0
médiane,30.0


Distribution du nombre de CV pertinents par Job


,nombre_jobs
taille_liste,
30,2500


Nombre d'occurrences de couples (job_id, resume_id) dupliquées : 0


## 11. Synthèse automatique et sauvegarde

Les principaux indicateurs d'audit sont regroupés dans un dictionnaire sérialisable puis écrits
dans `results/step1_dataset_audit.json`. Cette sortie décrit les données observées, sans nettoyage.

In [11]:
n_unique_roles = int(resumes_df["role"].nunique(dropna=True))
n_industries = int(pd.concat([resumes_df["industry"], jobs_df["industry"]]).nunique(dropna=True))
missing_values_total = int(sum(series.sum() for series in missing_by_table.values()))
duplicate_rows_total = int(sum(duplicate_rows.values()))

audit_summary = {
    "dataset_id": DATASET_ID,
    "n_resumes": n_resumes,
    "n_jobs": n_jobs,
    "n_match_records": n_match_records,
    "n_positive_pairs": n_positive_pairs,
    "n_unique_roles": n_unique_roles,
    "n_industries": n_industries,
    "n_unique_resume_skills": n_unique_resume_skills,
    "n_unique_job_skills": n_unique_job_skills,
    "missing_values_total": missing_values_total,
    "duplicate_rows_total": duplicate_rows_total,
    "invalid_job_references": invalid_job_references,
    "invalid_resume_references": invalid_resume_references,
    "duplicate_positive_pairs": n_duplicate_pairs,
}

output_path = RESULTS_DIR / "step1_dataset_audit.json"
with output_path.open("w", encoding="utf-8") as file:
    json.dump(audit_summary, file, indent=2, ensure_ascii=False)

display(pd.Series(audit_summary, name="valeur").to_frame())
print(f"Synthèse sauvegardée dans : {output_path}")

,valeur
dataset_id,michaelozon/candidate-matching-synthetic
n_resumes,10000
n_jobs,2500
n_match_records,2500
n_positive_pairs,75000
n_unique_roles,24
n_industries,10
n_unique_resume_skills,73
n_unique_job_skills,73
missing_values_total,0


Synthèse sauvegardée dans : results\step1_dataset_audit.json
